In [3]:
"""
Evaluate the accuracy of a Hugging Face image classification model.

Model folder (MODEL_DIR) must contain:
    config.json, preprocessor_config.json, model.safetensors

Test folder layout (folder name = class label):
    test_data/
        real/  img1.jpg, img2.jpg, ...
        fake/  img1.jpg, img2.jpg, ...
"""

from pathlib import Path

import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------- 1. Settings (change these) ----------
MODEL_DIR = r"E:\Hackathon\Build With Bharat 4.0\Asli-Kavach\backend\model\Video"   # folder with config.json, model.safetensors...
TEST_DIR = r"E:\Hackathon\Build With Bharat 4.0\Asli-Kavach\backend\test_scripts\test_content"            # folder with one sub-folder per class
BATCH_SIZE = 16
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------- 2. Load the model and its image processor ----------
processor = AutoImageProcessor.from_pretrained(MODEL_DIR)
model = AutoModelForImageClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()  # testing mode

id2label = model.config.id2label                                   # {0: "fake", 1: "real"}
label2id = {name.lower(): idx for name, idx in model.config.label2id.items()}
print(f"Model labels: {id2label}")

# ---------- 3. Collect test images and their true labels ----------
samples = []  # list of (image_path, true_label_id)

for class_folder in sorted(Path(TEST_DIR).iterdir()):
    if not class_folder.is_dir():
        continue

    label_id = label2id.get(class_folder.name.lower())
    if label_id is None:
        raise ValueError(
            f"Folder '{class_folder.name}' does not match any model label: "
            f"{list(id2label.values())}. Rename the folder to match."
        )

    for image_path in class_folder.iterdir():
        if image_path.suffix.lower() in IMAGE_EXTENSIONS:
            samples.append((image_path, label_id))

print(f"Found {len(samples)} test images")

# ---------- 4. Run predictions in batches ----------
all_preds = []
all_labels = []

with torch.no_grad():  # no training, so no gradients needed
    for start in range(0, len(samples), BATCH_SIZE):
        batch = samples[start:start + BATCH_SIZE]

        images = [Image.open(path).convert("RGB") for path, _ in batch]
        inputs = processor(images=images, return_tensors="pt").to(device)

        logits = model(**inputs).logits
        preds = logits.argmax(dim=-1)  # class with the highest score

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(label for _, label in batch)

        print(f"Processed {min(start + BATCH_SIZE, len(samples))}/{len(samples)}", end="\r")

# ---------- 5. Show the results ----------
label_ids = sorted(id2label.keys())
class_names = [id2label[i] for i in label_ids]

accuracy = accuracy_score(all_labels, all_preds)
print(f"\n\nAccuracy: {accuracy * 100:.2f}%\n")

print("Per-class report (precision / recall / F1):")
print(classification_report(all_labels, all_preds, labels=label_ids,
                            target_names=class_names, zero_division=0))

print("Confusion matrix (rows = true class, columns = predicted class):")
print(confusion_matrix(all_labels, all_preds, labels=label_ids))

Using device: cuda


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 208.98it/s]


Model labels: {0: 'FAKE', 1: 'REAL'}
Found 4 test images
Processed 4/4

Accuracy: 75.00%

Per-class report (precision / recall / F1):
              precision    recall  f1-score   support

        FAKE       0.67      1.00      0.80         2
        REAL       1.00      0.50      0.67         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4

Confusion matrix (rows = true class, columns = predicted class):
[[2 0]
 [1 1]]
